<a href="https://colab.research.google.com/github/jmoussa99/football-ai/blob/main/football_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Football AI

---

[![GitHub](https://badges.aleen42.com/src/github.svg)](https://github.com/roboflow/sports)
[![Roboflow](https://raw.githubusercontent.com/roboflow-ai/notebooks/main/assets/badges/roboflow-blogpost.svg)](https://blog.roboflow.com/camera-calibration-sports-computer-vision/)
[![YouTube](https://badges.aleen42.com/src/youtube.svg)](https://www.youtube.com/watch?v=aBVGKoNZQUw)

Let's build a Football AI system to dig deeper into match stats! We'll use computer vision and machine learning to track players, determine which team is which, and even calculate stuff like ball possession and speed. This tutorial is perfect if you want to get hands-on with sports analytics and see how AI can take your football analysis to the next level.

![football AI diagram](https://media.roboflow.com/notebooks/examples/football-ai-diagram.png)

## Before you start

Use a fresh Colab session. If you ran the old installation cells, choose
**Runtime → Disconnect and delete runtime** first. A GPU runtime is recommended;
the notebook also runs on CPU, with slower inference.

Run the dependency installation once, restart the session,
then continue from **Mount Google Drive**. Set your secrets and video path below.

### Configure your API keys

In Colab's **Secrets** panel (🔑), add `ROBOFLOW_API_KEY` from your
[Roboflow settings](https://app.roboflow.com/settings/api) and enable notebook access.
You can also set it as an environment variable.

`HF_TOKEN` is optional for the public SigLIP model. If you use a
[Hugging Face token](https://huggingface.co/settings/tokens), enable notebook access
for that secret too.

In [ ]:
import os
from google.colab import userdata

def get_notebook_secret(name, required=False):
    value = os.environ.get(name)
    if not value:
        try:
            value = userdata.get(name)
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError) as exc:
            if required:
                raise RuntimeError(
                    f"Add {name} in Colab Secrets and enable notebook access."
                ) from exc
    if required and not value:
        raise RuntimeError(f"Set {name} in Colab Secrets or the environment.")
    if value:
        os.environ[name] = value
    return value

ROBOFLOW_API_KEY = get_notebook_secret("ROBOFLOW_API_KEY", required=True)
get_notebook_secret("HF_TOKEN")

### Select the runtime

Choose **Runtime → Change runtime type → T4 GPU** (or another available GPU).
The check below reports whether NVIDIA hardware is available. PyTorch's usable
device is checked separately after installation.

In [ ]:
import shutil
import subprocess

HAS_NVIDIA_GPU = bool(shutil.which("nvidia-smi")) and subprocess.run(
    ["nvidia-smi"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
).returncode == 0
if HAS_NVIDIA_GPU:
    subprocess.run(["nvidia-smi"], check=True)
else:
    print("No NVIDIA GPU detected. The notebook will use CPU inference.")

## Install dependencies

Install the packages together so pip resolves their requirements in one pass.
These versions retain the `sv.ByteTrack` and SigLIP APIs used below. The Inference
GPU package is selected when NVIDIA hardware is available. Colab's existing
PyTorch installation is preserved, and Pillow is resolved with Inference rather
than downgraded in a later cell.

In [ ]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from tempfile import TemporaryDirectory

# Inference 0.38 supports Python 3.9–3.12, including this notebook's Colab runtime.
if not (3, 9) <= sys.version_info[:2] < (3, 13):
    raise RuntimeError("Use a Colab Python 3.12 runtime for these dependencies.")

inference_package = "inference-gpu" if HAS_NVIDIA_GPU else "inference"
requirements = [
    f"{inference_package}==0.38.0",
    "supervision==0.25.1",
    "transformers==4.48.3",
    "pillow>=10.1,<11",
    "protobuf>=3.20.3,<6",
    "sports @ git+https://github.com/roboflow/sports.git@42c80c06b6b65a7f89455b89fe31cdf4c38ba227",
]

# Preserve Colab's CUDA build and the pandas/requests versions used by Colab itself.
constraints = []
for package in ("torch", "torchvision", "pandas", "requests"):
    try:
        constraints.append(f"{package}=={version(package)}")
    except PackageNotFoundError:
        pass

with TemporaryDirectory() as directory:
    constraint_path = Path(directory) / "constraints.txt"
    constraint_path.write_text("\n".join(constraints))
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "--constraint", str(constraint_path), *requirements,
    ])

print("Installation complete. Restart the session once")
print("and continue from 'Mount Google Drive'; do not rerun the old install cells.")

The sports utilities are installed at a fixed repository revision by the cell
above. No separate GitHub installation is needed.

If you restart the session after installation, continue below; credentials are
loaded again in the model setup cell.

Check the installed package versions before loading the models.

In [ ]:
from importlib.metadata import version

for package in ("supervision", "transformers", "pillow", "torch"):
    print(f"{package}: {version(package)}")

In [ ]:
# Mount Google Drive (continue here after restarting the session).
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os
import torch
import onnxruntime as ort

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
available_providers = ort.get_available_providers()
providers = ["CPUExecutionProvider"]
if DEVICE == "cuda" and "CUDAExecutionProvider" in available_providers:
    providers.insert(0, "CUDAExecutionProvider")

# Set these before importing inference. Keep CPU fallback for detection models.
os.environ["ONNXRUNTIME_EXECUTION_PROVIDERS"] = "[" + ",".join(providers) + "]"
for model in ("CLIP", "SAM", "SAM2", "OWLV2", "GAZE", "DOCTR", "TROCR",
              "GROUNDINGDINO", "YOLO_WORLD"):
    os.environ[f"CORE_MODEL_{model}_ENABLED"] = "False"

print(f"Team classifier device: {DEVICE}")
print(f"Detection execution providers: {providers}")
if DEVICE == "cpu":
    print("CUDA is unavailable in PyTorch; continuing on CPU.")

Pillow was installed with the other dependencies. Continue to model loading;
there is no separate Pillow downgrade.

## ball, player, goalkeeper and referee detection

In [ ]:
# Reload secrets here too, so this cell works after a session restart.
import os
from google.colab import userdata
from inference import get_model

ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY")
if not ROBOFLOW_API_KEY:
    try:
        ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError) as exc:
        raise RuntimeError(
            "Add ROBOFLOW_API_KEY in Colab Secrets and enable notebook access."
        ) from exc
if not ROBOFLOW_API_KEY:
    raise RuntimeError("ROBOFLOW_API_KEY is required to load the detection models.")
os.environ["ROBOFLOW_API_KEY"] = ROBOFLOW_API_KEY

if not os.environ.get("HF_TOKEN"):
    try:
        hf_token = userdata.get("HF_TOKEN")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        hf_token = None
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token

PLAYER_DETECTION_MODEL_ID = "football-players-detection-3zvbc/11"
FIELD_DETECTION_MODEL_ID = "football-field-detection-f07vi/14"
PLAYER_DETECTION_MODEL = get_model(
    model_id=PLAYER_DETECTION_MODEL_ID, api_key=ROBOFLOW_API_KEY)
FIELD_DETECTION_MODEL = get_model(
    model_id=FIELD_DETECTION_MODEL_ID, api_key=ROBOFLOW_API_KEY)

## Choose the video and train the team classifier

Set `SOURCE_VIDEO_PATH` to your video in Google Drive (or an uploaded Colab file).
Only player crops are used for training. Both teams should be visible in the
sampled frames; lower `STRIDE` for a short clip with too few detections.

In [ ]:
from pathlib import Path
import numpy as np
import supervision as sv

SOURCE_VIDEO_PATH = "/content/drive/MyDrive/Soccer_Video_Analysis/alabama-vs-northwestern-new-left-side-dribble.mp4"
if not Path(SOURCE_VIDEO_PATH).is_file():
    raise FileNotFoundError(
        f"Video not found: {SOURCE_VIDEO_PATH}. Mount Drive and check the path.")

video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
if not np.isfinite(video_info.fps) or video_info.fps <= 0:
    raise ValueError("The input video must have a valid, positive frame rate.")
if video_info.total_frames <= 0:
    raise ValueError("The input video contains no readable frames.")
OUTPUT_VIDEO_PATH = str(Path(SOURCE_VIDEO_PATH).with_name(
    Path(SOURCE_VIDEO_PATH).stem + "_output_tracking.mp4"))
print(video_info)

In [ ]:
import numpy as np
import supervision as sv
from tqdm.auto import tqdm
from sports.common.team import TeamClassifier

BALL_ID = 0
GOALKEEPER_ID = 1
PLAYER_ID = 2
REFEREE_ID = 3
STRIDE = 30  # Sample every STRIDE frames for classifier training.

def crop_players(frame, detections):
    """Return aligned detections/crops, skipping empty or invalid image regions."""
    height, width = frame.shape[:2]
    indices, crops = [], []
    for index, box in enumerate(detections.xyxy):
        if not np.isfinite(box).all():
            continue
        x1, y1, x2, y2 = np.clip(box, [0, 0, 0, 0], [width, height, width, height]).astype(int)
        if x2 <= x1 or y2 <= y1:
            continue
        indices.append(index)
        crops.append(frame[y1:y2, x1:x2])
    return detections[np.asarray(indices, dtype=int)], crops

crops = []
frame_generator = sv.get_video_frames_generator(
    source_path=SOURCE_VIDEO_PATH, stride=STRIDE)
for frame in tqdm(frame_generator, desc="Collecting player crops"):
    result = PLAYER_DETECTION_MODEL.infer(frame, confidence=0.3)[0]
    detections = sv.Detections.from_inference(result)
    players = detections[detections.class_id == PLAYER_ID].with_nms(threshold=0.5)
    _, players_crops = crop_players(frame, players)
    crops.extend(players_crops)

# Three-dimensional UMAP needs more than four samples for spectral initialization.
if len(crops) < 6:
    raise ValueError(
        f"Only {len(crops)} valid player crops found. Use a longer video or lower STRIDE.")
print(f"Collected {len(crops)} player crops.")

In [ ]:
# Train once. DEVICE comes from torch.cuda.is_available(), not a hard-coded GPU.
team_classifier = TeamClassifier(device=DEVICE)
team_classifier.fit(crops)

In [ ]:
import cv2
import numpy as np
from scipy.spatial.distance import cdist

def draw_all_connections_on_frame(
    frame_image: np.ndarray,
    detections: sv.Detections,
    color: sv.Color,
    line_thickness: int = 1,
    num_neighbors: int = 2,
) -> np.ndarray:
    if len(detections) < 2 or num_neighbors <= 0:
        return frame_image
    output_image = frame_image.copy()
    points = detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER).astype(int)
    distances = cdist(points, points)
    np.fill_diagonal(distances, np.inf)
    drawn_connections = set()
    count = min(num_neighbors, len(points) - 1)
    for index, neighbors in enumerate(np.argsort(distances, axis=1)[:, :count]):
        for neighbor in neighbors:
            connection = tuple(sorted((index, int(neighbor))))
            if connection not in drawn_connections:
                cv2.line(output_image, tuple(points[index]), tuple(points[neighbor]),
                         color.as_bgr(), line_thickness, cv2.LINE_AA)
                drawn_connections.add(connection)
    return output_image

In [ ]:
UNKNOWN_TEAM_ID = 3  # Neutral gray; referee annotations use class 2.

def resolve_goalkeepers_team_id(players, goalkeepers):
    """Assign by nearest team centroid when both teams are visible."""
    team_ids = np.full(len(goalkeepers), UNKNOWN_TEAM_ID, dtype=int)
    if not len(goalkeepers) or not len(players):
        return team_ids
    players_xy = players.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    team_0 = players_xy[players.class_id == 0]
    team_1 = players_xy[players.class_id == 1]
    if not len(team_0) or not len(team_1):
        return team_ids
    centroids = np.array([team_0.mean(axis=0), team_1.mean(axis=0)])
    goalkeepers_xy = goalkeepers.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    distances = np.linalg.norm(goalkeepers_xy[:, None, :] - centroids[None, :, :], axis=2)
    return np.argmin(distances, axis=1).astype(int)

In [ ]:
from sports.common.view import ViewTransformer
from sports.configs.soccer import SoccerPitchConfiguration

CONFIG = SoccerPitchConfiguration()

def get_pitch_transformer(key_points, config=CONFIG):
    """Return None for missing/insufficient keypoints or a degenerate homography."""
    if key_points is None or key_points.confidence is None or len(key_points.xy) == 0:
        return None
    xy = key_points.xy[0]
    if len(xy) == 0:
        return None
    confidence = key_points.confidence[0]
    vertices = np.asarray(config.vertices, dtype=np.float32)
    if len(xy) != len(vertices) or len(confidence) != len(xy):
        raise ValueError("Field-model keypoints do not match the configured pitch vertices.")
    valid = (confidence > 0.5) & np.isfinite(confidence) & np.isfinite(xy).all(axis=1)
    if np.count_nonzero(valid) < 4:
        return None
    source = xy[valid]
    target = vertices[valid] / 100.0
    if (np.linalg.matrix_rank(source - source.mean(axis=0)) < 2
            or np.linalg.matrix_rank(target - target.mean(axis=0)) < 2):
        return None
    try:
        # SoccerPitchConfiguration vertices are centimeters; transform to meters.
        transformer = ViewTransformer(source=source, target=target)
    except (ValueError, cv2.error):
        return None
    if not np.isfinite(transformer.m).all() or np.linalg.matrix_rank(transformer.m) < 3:
        return None
    return transformer

class PlayerSpeedEstimator:
    """Measure displacement between sampled frames in pixels and pitch meters."""
    def __init__(self, fps, sample_stride=30):
        if not np.isfinite(fps) or fps <= 0 or sample_stride < 1:
            raise ValueError("fps must be positive and sample_stride must be at least 1.")
        self.fps = fps
        self.sample_stride = sample_stride
        self.last_sample_frame = None
        self.pixel_positions = {}
        self.meter_positions = {}
        self.speeds_pps = {}
        self.speeds_mps = {}

    def update(self, frame_index, detections, transformer=None):
        tracker_ids = detections.tracker_id
        if tracker_ids is None:
            tracker_ids = np.empty(0, dtype=int)
        visible_ids = set(tracker_ids)
        # A missed detection breaks the measurement interval for that track.
        for history in (self.pixel_positions, self.meter_positions,
                        self.speeds_pps, self.speeds_mps):
            for tracker_id in list(history):
                if tracker_id not in visible_ids:
                    del history[tracker_id]
        for tracker_id in tracker_ids:
            self.speeds_pps.setdefault(tracker_id, 0.0)
            self.speeds_mps.setdefault(tracker_id, None)
        if frame_index % self.sample_stride:
            return self.speeds_pps, self.speeds_mps

        points = detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
        pixels = {tracker_id: point.copy() for tracker_id, point in zip(tracker_ids, points)}
        meters = {}
        if transformer is not None and len(points):
            transformed = transformer.transform_points(points)
            meters = {tracker_id: point.copy() for tracker_id, point in zip(tracker_ids, transformed)
                      if np.isfinite(point).all()}

        elapsed = None if self.last_sample_frame is None else (frame_index - self.last_sample_frame) / self.fps
        for tracker_id in tracker_ids:
            self.speeds_pps[tracker_id] = 0.0
            self.speeds_mps[tracker_id] = None
            if elapsed is not None and elapsed > 0:
                if tracker_id in pixels and tracker_id in self.pixel_positions:
                    self.speeds_pps[tracker_id] = float(np.linalg.norm(
                        pixels[tracker_id] - self.pixel_positions[tracker_id]) / elapsed)
                if tracker_id in meters and tracker_id in self.meter_positions:
                    self.speeds_mps[tracker_id] = float(np.linalg.norm(
                        meters[tracker_id] - self.meter_positions[tracker_id]) / elapsed)

        self.pixel_positions = pixels
        self.meter_positions = meters
        self.last_sample_frame = frame_index
        return self.speeds_pps, self.speeds_mps

## Process the video

This single pass tracks players and goalkeepers, assigns team colors, annotates
the ball and referees, and saves the result beside the source video.

`SPEED_STRIDE` controls the interval between speed measurements; every video frame
is still tracked and written. Metric speed requires usable pitch keypoints at
both measurement frames. Pixel speed continues when calibration is unavailable.
The metric estimate assumes the configured pitch dimensions match the real field.

In [ ]:
SPEED_STRIDE = 30
DRAW_TEAM_CONNECTIONS = False

palette = sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700', '#808080'])
ellipse_annotator = sv.EllipseAnnotator(color=palette, thickness=2)
triangle_annotator = sv.TriangleAnnotator(
    color=sv.Color.from_hex('#FFD700'), base=25, height=21, outline_thickness=1)
label_annotator = sv.LabelAnnotator(
    color=palette, text_color=sv.Color.from_hex('#000000'),
    text_position=sv.Position.BOTTOM_CENTER)

tracker = sv.ByteTrack(frame_rate=int(round(video_info.fps)))
speed_estimator = PlayerSpeedEstimator(video_info.fps, sample_stride=SPEED_STRIDE)
frames_without_calibration = 0
processed_frames = 0

with sv.VideoSink(OUTPUT_VIDEO_PATH, video_info) as sink:
    frames = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
    for frame_index, frame in tqdm(enumerate(frames), total=video_info.total_frames,
                                   desc="Tracking video"):
        result = PLAYER_DETECTION_MODEL.infer(frame, confidence=0.3)[0]
        detections = sv.Detections.from_inference(result)
        ball_detections = detections[detections.class_id == BALL_ID]
        ball_detections.xyxy = sv.pad_boxes(xyxy=ball_detections.xyxy, px=10)

        people = detections[detections.class_id != BALL_ID].with_nms(
            threshold=0.5, class_agnostic=True)
        people = tracker.update_with_detections(detections=people)
        goalkeepers = people[people.class_id == GOALKEEPER_ID]
        players = people[people.class_id == PLAYER_ID]
        referees = people[people.class_id == REFEREE_ID]

        players, player_crops = crop_players(frame, players)
        players.class_id = (np.asarray(team_classifier.predict(player_crops), dtype=int)
                            if player_crops else np.empty(0, dtype=int))
        goalkeepers.class_id = resolve_goalkeepers_team_id(players, goalkeepers)
        referees.class_id = np.full(len(referees), 2, dtype=int)

        participants = sv.Detections.merge([players, goalkeepers])
        final_detections = sv.Detections.merge([participants, referees])

        transformer = None
        if frame_index % SPEED_STRIDE == 0:
            field_result = FIELD_DETECTION_MODEL.infer(frame, confidence=0.3)[0]
            transformer = get_pitch_transformer(sv.KeyPoints.from_inference(field_result))
            if transformer is None:
                frames_without_calibration += 1
        player_speeds_pps, player_speeds_mps = speed_estimator.update(
            frame_index, participants, transformer)

        labels = []
        tracker_ids = final_detections.tracker_id
        if tracker_ids is not None:
            for tracker_id in tracker_ids:
                label = f"#{tracker_id}"
                if tracker_id in player_speeds_pps:
                    label += f" P: {player_speeds_pps[tracker_id]:.1f} px/s"
                    speed_mps = player_speeds_mps[tracker_id]
                    if speed_mps is not None:
                        label += f" | M: {speed_mps:.1f} m/s"
                labels.append(label)

        annotated_frame = ellipse_annotator.annotate(frame.copy(), final_detections)
        annotated_frame = label_annotator.annotate(
            annotated_frame, final_detections, labels=labels)
        annotated_frame = triangle_annotator.annotate(annotated_frame, ball_detections)
        if DRAW_TEAM_CONNECTIONS:
            for team_id in (0, 1):
                annotated_frame = draw_all_connections_on_frame(
                    annotated_frame, players[players.class_id == team_id],
                    color=palette.by_idx(team_id))
        sink.write_frame(annotated_frame)
        processed_frames += 1

if not processed_frames:
    raise RuntimeError("No video frames were decoded. Check the input video.")
print(f"Processing complete. Output video saved to {OUTPUT_VIDEO_PATH}")
if frames_without_calibration:
    print(f"Pitch calibration unavailable at {frames_without_calibration} speed samples; "
          "metric speed is omitted for those intervals.")

In [ ]:
print(f"Annotated video: {OUTPUT_VIDEO_PATH}")
# To download the result to your computer, run:
# from google.colab import files
# files.download(OUTPUT_VIDEO_PATH)